In [ ]:
import uproot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# === Load metadata table ===
bigtable = pd.read_csv("rsidis_bigtable_pass0.csv")

ebeam = 8.5831
x = 0.25
Q2 = 3.3
z = 0.36
thpq = 2.0
run_type = "PI-SIDIS"

# === Apply kinematic filters ===
mask = (
    (bigtable["run_type"] == run_type)
    & (bigtable["ebeam"] == ebeam)
    & (bigtable["hms_p"] < 0)
    & (bigtable["x"] == x)
    & (bigtable["Q2"] == Q2)
    & (bigtable["z"] == z)
    & (bigtable["thpq"] == thpq)
)

selected_runs = bigtable[mask]["run"].tolist()

print("Selected runs:", selected_runs)

# === Collect data from selected ROOT files ===
branch_name = "Q2"   # <-- change this

all_values = []

for run in selected_runs:
    filename = f"../pass0/_{run}.root"
    try:
        with uproot.open(filename) as f:
            tree = f["T"]   # adjust name, e.g., "T", "tree", "events"
            arr = tree[branch_name].array(library="np")
            all_values.append(arr)
    except Exception as e:
        print(f"Error reading run {run}: {e}")

# combine all arrays
all_values = np.concatenate(all_values)

# === Plot distribution ===
plt.figure(figsize=(8,5))
plt.hist(all_values, bins=100, histtype="stepfilled", alpha=0.6)
plt.xlabel(branch_name)
plt.ylabel("Counts")
plt.title(f"Distribution of {branch_name} for selected runs")
plt.grid(alpha=0.3)
plt.show()